In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

In [ ]:

# 1. Setup Paths
base_path = "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod"

try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Add Include Path so classes can find each other's headers
ROOT.gInterpreter.AddIncludePath(base_path)

# 4. Explicit Compilation and Loading Function
def load_custom_class(class_name):
    source_file = os.path.join(base_path, f"{class_name}.cpp")
    header_file = os.path.join(base_path, f"{class_name}.h")
    so_file = os.path.join(base_path, f"{class_name}_cpp.so")
    
    # Compile with 'k' (keep), 'O' (optimize), 'f' (force)
    # Force is useful to ensure the symbol table is rebuilt correctly
    status = ROOT.gSystem.CompileMacro(source_file, "kOf")
    
    if status >= 0:
        # CRITICAL: Manually load the shared library to resolve symbols
        if ROOT.gSystem.Load(so_file) < 0:
            print(f"⚠️  Compiled {class_name} but failed to load {so_file}")
            return False
            
        # Declare the header to the interpreter to help dictionary lookup
        ROOT.gInterpreter.Declare(f'#include "{header_file}"')
        print(f"📦 Successfully compiled and linked: {class_name}")
        return True
    else:
        print(f"❌ Failed to compile: {class_name}")
        return False

# 5. Execute Sequence
# We must load PhysdEdx first as Hypfit depends on it
if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        try:
            # Re-declaring just to be safe before instantiation
            ROOT.gInterpreter.ProcessLine(f'#include "{os.path.join(base_path, "Hypfit.h")}"')
            
            # Instantiate
            h_fit = ROOT.Hypfit()
            print("🚀 Success! Hypfit object initialized and linked.")
        except Exception as e:
            print(f"❌ Error during instantiation: {e}")
            # Final fallback: access via C++ global pointer if Python attribute fails
            ROOT.gInterpreter.ProcessLine("Hypfit* h_fit_ptr = new Hypfit();")
            h_fit = ROOT.h_fit_ptr
            print("🚀 Success! Hypfit object initialized via Global Pointer fallback.")
            

# Load DataFrames

In [ ]:
## Check keys in each file

keys2load = ["cc1pi","nudf", "hdr", "pot", "histpotdf", "hit2"]
cv_df = load_df("/scratch/7DayLifetime/lpelegrina/hit2df_cv.df", keys2load, 5)
ccalm_df = load_df("/scratch/7DayLifetime/lpelegrina/hit2df_ccalm.df", keys2load, 5)
keys2load = ["pandora","nudf", "hdr", "pot", "histpotdf", "hit2"]
ccalp_df = load_df("/scratch/7DayLifetime/lpelegrina/hit2df_ccalp.df", keys2load, 5)

# Test background composition

In [ ]:
# 2. Extract the DataFrame from the dictionary and rename the column
# Replace 'pandora' with the actual key found in step 1 if it's different
cv_df_actual = cv_df['cc1pi'].rename(columns={"cc1pi": "pandora"})
ccalm_df_actual = ccalm_df['cc1pi'].rename(columns={"cc1pi": "pandora"})

# 3. If you want to put them back into the dictionary structure:
cv_df['pandora'] = cv_df_actual
ccalm_df['pandora'] = ccalm_df_actual

In [ ]:

from analysis_village.cc1pi.Constants import CTE as CTE
def exiting_pfp_mask(df):
    xmin = -200 + CTE.min_distance_to_consider_contained
    xmax = 200 - CTE.min_distance_to_consider_contained
    ymin = -200 + CTE.min_distance_to_consider_contained
    ymax = 200 - CTE.min_distance_to_consider_contained
    zmin = CTE.min_distance_to_consider_contained
    zmax = 500 - CTE.min_distance_to_consider_contained
    
    not_in_fv_start = (df.pfp.trk.start.x < xmin) | (df.pfp.trk.start.x > xmax) | (df.pfp.trk.start.y < ymin) | (df.pfp.trk.start.y > ymax) | (df.pfp.trk.start.z < zmin) | (df.pfp.trk.start.z  > zmax)
    not_in_fv_end = (df.pfp.trk.end.x < xmin) | (df.pfp.trk.end.x > xmax) | (df.pfp.trk.end.y < ymin) | (df.pfp.trk.end.y > ymax) | (df.pfp.trk.end.z < zmin) | (df.pfp.trk.end.z  > zmax)
    return not_in_fv_start | not_in_fv_end

In [ ]:
def filter_hits_by_pfp(hit_dfs):
    """
    Filters 'hit2' DataFrames based on muon PDG (13) in 'pandora' DataFrames.
    Returns a new dictionary with the same keys.
    """
    filtered_output = {}
    pdg_col = ('pfp', 'trk', 'truth', 'p', 'pdg', '')

    for key, data_dict in hit_dfs.items():
        print(f"Filtering hits for: {key}")
        
        # 1. Access the internal DataFrames
        df_pandora = data_dict["pandora"]
        df_hits = data_dict["hit2"]

        # 2. Create the muon mask on the PFP-level DataFrame
        # Check if column exists to avoid KeyErrors
        if pdg_col not in df_pandora.columns:
            print(f"  Warning: PDG column not found in {key}['pandora']. Skipping.")
            filtered_output[key] = data_dict
            continue

        pfp_mask = abs(df_pandora[pdg_col]) == 211
        candidate_mask = (df_pandora.pfp.trk.len > 10) & (df_pandora.pfp.trk.chi2pid.I2.chi2_muon < 20) & (df_pandora.pfp.trk.chi2pid.I2.chi2_proton > 85)
        contained_mask = ~exiting_pfp_mask(df_pandora)
        stopping_mask = ~df_pandora.pfp.trk.truth.p.end_process.isin([3, 45])
        muon_indices = df_pandora[pfp_mask & candidate_mask & stopping_mask & contained_mask].index
        #muon_indices = df_pandora[pfp_mask & candidate_mask & contained_mask].index

        # 3. Align the hit index (5 levels) with the PFP index (4 levels)
        # We drop the last level: 'rec.slc.reco.pfp.trk.calo.2.points..index'
        hit_index_aligned = df_hits.index.droplevel(-1)

        # 4. Filter the hits
        is_muon_hit = hit_index_aligned.isin(muon_indices)
        df_hit2_filtered = df_hits[is_muon_hit]

        # 5. Store in the new structure
        # We make a copy of the dictionary to avoid modifying the original hit_dfs
        df_hit2_filtered
        
        filtered_output[key] = df_hit2_filtered
        
        print(f"  Done. {len(df_hit2_filtered)} hits kept out of {len(df_hits)}.")

    return filtered_output

# Usage:
#

hit_dfs = {
    "cv": cv_df,
    "ccalp": ccalp_df,
    "ccalm": ccalm_df,
}

filtered_inelastic_hit_dfs = filter_hits_by_pfp(hit_dfs)
#filtered_stopping_hit_dfs = filter_hits_by_pfp(hit_dfs)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Setup intervals from 0 to 30 with 1cm steps
# This creates [(0, 1), (1, 2), ..., (29, 30)]
intervals = [(i, i+1) for i in range(30)]

# Constants for plotting
dedx_col = "dedx"
rr_col = "rr"
samples = [
    ('cv', 'black', 'CV'),
    ('ccalp', 'red', 'CCAL +1$\sigma$'),
    ('ccalm', 'blue', 'CCAL -1$\sigma$')
]

# Create a directory for the plots if you want to save them
# os.makedirs("dedx_histograms", exist_ok=True)

for (start, end) in intervals:
    plt.figure(figsize=(8, 5))
    
    found_data = False
    for label, color, display_name in samples:
        df = filtered_stopping_hit_dfs[label]
        
        # Filter for the specific RR bin
        mask = (df[rr_col] >= start) & (df[rr_col] < end)
        dedx_values = df.loc[mask, dedx_col]
        
        if not dedx_values.empty:
            found_data = True
            plt.hist(dedx_values, bins=60, range=(0, 5), 
                     histtype='step', linewidth=1.5, color=color, 
                     label=display_name, density=True)
    
    if not found_data:
        plt.close() # Skip empty plots
        continue

    # Formatting
    plt.title(f"dE/dx Distribution | RR Interval: [{start}, {end}] cm", fontsize=13)
    plt.xlabel(r"dE/dx [MeV/cm]", fontsize=12)
    plt.ylabel("Normalized Unit Area", fontsize=12)
    plt.grid(axis='y', linestyle=':', alpha=0.4)
    plt.legend(loc='upper right', frameon=True)
    
    # Optional: Save and close to manage memory
    # plt.savefig(f"dedx_histograms/rr_{start}_{end}.png")
    # plt.close() 
    
    plt.show()

In [ ]:
def calculate_hypfit_p(track_id, best_plane, hit_df, target_pdg=211, cleaning="none"):
    try:
        # 1. Extract and sort hits
        track_hits = hit_df.loc[track_id]
        # 2. Strict type casting for C++ 
        # Fix: Handle cases where best_plane is a Series or a scalar
        if hasattr(best_plane, "iloc"):
            c_plane = int(best_plane.iloc[0])
        else:
            c_plane = int(best_plane)
            
        c_pdg = int(target_pdg)
        c_cleaning = str(cleaning)

        # Convert to contiguous float64 numpy arrays
        rr    = np.ascontiguousarray(track_hits['rr'].to_numpy(dtype=np.float64))
        dedx  = np.ascontiguousarray(track_hits['dedx'].to_numpy(dtype=np.float64))
        pitch = np.ascontiguousarray(track_hits['pitch'].to_numpy(dtype=np.float64))
        
        # Build mask: keep only entries where ALL values are finite
        mask = np.isfinite(rr) & np.isfinite(dedx) & np.isfinite(pitch)
        
        # Apply mask consistently to all arrays
        rr    = rr[mask]
        dedx  = dedx[mask]
        pitch = pitch[mask]
        
        # Now create ROOT vectors
        v_rr    = ROOT.std.vector('double')(rr)
        v_dedx  = ROOT.std.vector('double')(dedx)
        v_pitch = ROOT.std.vector('double')(pitch)

        reco_p = -10000
        reco_p = h_fit.GetTLExtensionP(c_pdg, v_rr, v_dedx, v_pitch, c_plane, c_cleaning)
        # Explicitly clear/delete to help PyROOT
        if 'v_rr' in locals():
            v_rr.clear()
            v_dedx.clear()
            v_pitch.clear()
            # Explicitly delete the Python reference to the C++ object
            del v_rr, v_dedx, v_pitch
            
        return reco_p
           
    except KeyError:
        return -1.0
    except Exception as e:
        print(f"Error for track {track_id}: {e}")
        return -1.0

    finally:
        # 1. Clear the vectors
        if 'v_rr' in locals():
            v_rr.clear()
            v_dedx.clear()
            v_pitch.clear()
        
        # 2. Force ROOT to cleanup any temporary objects/TGraphs/TF1s 
        # created during the C++ call
        ROOT.gDirectory.Clear()
        
        # 3. Specifically clear the global list of functions to prevent 
        # accumulation of TF1 objects created in C++
        ROOT.gROOT.GetListOfFunctions().Clear()

In [ ]:
for label in ['cv', 'ccalp', 'ccalm']:
    filtered_hit_dfs[label] = filtered_hit_dfs[label].sort_values('rr', ascending=True)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm.auto import tqdm # Progress bar

# 1. Initialize storage for momentum values
p_values = {
    'cv': [],
    'ccalp': [],
    'ccalm': []
}

unique_tracks = filtered_hit_dfs['cv'].index.droplevel(-1).unique()
rr_col = "rr"
dedx_col = "dedx"


#do_plot = (len(p_values['cv']) < 5) 
do_plot = False

# 2. Loop with progress bar (tqdm)
# Limit to 1000 tracks as requested
for trk_id in tqdm(unique_tracks[:10000], desc="Processing Tracks"):
    
    cv_data = None
    # We only plot the first few to avoid memory/display issues

    if do_plot:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, 
                                       gridspec_kw={'height_ratios': [3, 1]})
        plt.subplots_adjust(hspace=0.05)

    for label, color, linestyle in zip(['cv', 'ccalp', 'ccalm'], 
                                       ['black', 'red', 'blue'],
                                       ['-', '--', '--']):
        df_hits = filtered_hit_dfs[label]
        try:
            track_hits = df_hits.loc[trk_id].sort_values(by=rr_col)
            track_hits = track_hits[(track_hits.dedx > 0.5) & (track_hits.dedx < 20)]
            
            if track_hits.empty:
                continue

            reco_p = calculate_hypfit_p(
                track_id=trk_id, 
                best_plane=2, 
                hit_df=df_hits, 
                target_pdg=211, 
                cleaning="all"
            )
            
            # Store value if within requested range [0.13, 1]
            if 0.13 <= reco_p <= 1.0:
                p_values[label].append(reco_p)

            # --- Optional Plotting Logic ---
            if do_plot:
                x, y = track_hits[rr_col].values, track_hits[dedx_col].values
                ax1.plot(x, y, label=f"{label.upper()} (p={reco_p:.2f})", color=color, 
                         linestyle=linestyle, marker='o', markersize=3, alpha=0.7)
                if label == 'cv':
                    cv_data = (x, y)
                    ax2.axhline(1, color='black', linewidth=1)
                elif cv_data is not None:
                    cv_interp = np.interp(x, cv_data[0], cv_data[1])
                    ax2.plot(x, y / cv_interp, color=color, linestyle=linestyle, marker='o', markersize=3)
        
        except KeyError:
            pass # Silent skip for batch processing

    if do_plot:
        ax1.set_title(f"Track Details: {trk_id[1]}")
        ax1.legend(fontsize=8)
        plt.show()

# 3. Plot the final 3 histograms
plt.figure(figsize=(10, 6))

for label, color in [('cv', 'black'), ('ccalp', 'red'), ('ccalm', 'blue')]:
    plt.hist(p_values[label], bins=40, range=(0.13, 1.0), histtype='step', 
             linewidth=2, color=color, label=label.upper())

plt.title("Reconstructed Momentum Distribution (0.13 - 1.0 GeV/c)", fontsize=14)
plt.xlabel("Reco Momentum [GeV/c]", fontsize=12)
plt.ylabel("Counts", fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm.auto import tqdm # Progress bar

# 1. Initialize storage for momentum values
p_values = {
    'cv': [],
    'ccalp': [],
    'ccalm': []
}

unique_tracks = filtered_hit_dfs['cv'].index.droplevel(-1).unique()
rr_col = "rr"
dedx_col = "dedx"


#do_plot = (len(p_values['cv']) < 5) 
do_plot = False

# 2. Loop with progress bar (tqdm)
# Limit to 1000 tracks as requested
for trk_id in tqdm(unique_tracks[:10000], desc="Processing Tracks"):
    
    cv_data = None
    # We only plot the first few to avoid memory/display issues

    if do_plot:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, 
                                       gridspec_kw={'height_ratios': [3, 1]})
        plt.subplots_adjust(hspace=0.05)

    for label, color, linestyle in zip(['cv', 'ccalp', 'ccalm'], 
                                       ['black', 'red', 'blue'],
                                       ['-', '--', '--']):
        df_hits = filtered_hit_dfs[label]
        try:
            track_hits = df_hits.loc[trk_id].sort_values(by=rr_col)
            track_hits = track_hits[(track_hits.dedx > 0.5) & (track_hits.dedx < 20)]
            
            if track_hits.empty:
                continue

            reco_p = calculate_hypfit_p(
                track_id=trk_id, 
                best_plane=2, 
                hit_df=df_hits, 
                target_pdg=211, 
                cleaning="skip_only"
            )
            
            # Store value if within requested range [0.13, 1]
            if 0.13 <= reco_p <= 1.0:
                p_values[label].append(reco_p)

            # --- Optional Plotting Logic ---
            if do_plot:
                x, y = track_hits[rr_col].values, track_hits[dedx_col].values
                ax1.plot(x, y, label=f"{label.upper()} (p={reco_p:.2f})", color=color, 
                         linestyle=linestyle, marker='o', markersize=3, alpha=0.7)
                if label == 'cv':
                    cv_data = (x, y)
                    ax2.axhline(1, color='black', linewidth=1)
                elif cv_data is not None:
                    cv_interp = np.interp(x, cv_data[0], cv_data[1])
                    ax2.plot(x, y / cv_interp, color=color, linestyle=linestyle, marker='o', markersize=3)
        
        except KeyError:
            pass # Silent skip for batch processing

    if do_plot:
        ax1.set_title(f"Track Details: {trk_id[1]}")
        ax1.legend(fontsize=8)
        plt.show()

# 3. Plot the final 3 histograms
plt.figure(figsize=(10, 6))

for label, color in [('cv', 'black'), ('ccalp', 'red'), ('ccalm', 'blue')]:
    plt.hist(p_values[label], bins=40, range=(0.13, 1.0), histtype='step', 
             linewidth=2, color=color, label=label.upper())

plt.title("Reconstructed Momentum Distribution (0.13 - 1.0 GeV/c)", fontsize=14)
plt.xlabel("Reco Momentum [GeV/c]", fontsize=12)
plt.ylabel("Counts", fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm.auto import tqdm # Progress bar

# 1. Initialize storage for momentum values
p_values = {
    'cv': [],
    'ccalp': [],
    'ccalm': []
}

unique_tracks = filtered_hit_dfs['cv'].index.droplevel(-1).unique()
rr_col = "rr"
dedx_col = "dedx"


#do_plot = (len(p_values['cv']) < 5) 
do_plot = True

# 2. Loop with progress bar (tqdm)
# Limit to 1000 tracks as requested
for trk_id in tqdm(unique_tracks[:500], desc="Processing Tracks"):
    
    cv_data = None
    # We only plot the first few to avoid memory/display issues

    if do_plot:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, 
                                       gridspec_kw={'height_ratios': [3, 1]})
        plt.subplots_adjust(hspace=0.05)

    for label, color, linestyle in zip(['cv', 'ccalp', 'ccalm'], 
                                       ['black', 'red', 'blue'],
                                       ['-', '--', '--']):
        df_hits = filtered_inelastic_hit_dfs[label]
        try:
            track_hits = df_hits.loc[trk_id].sort_values(by=rr_col)
            track_hits = track_hits[(track_hits.dedx > 0.5) & (track_hits.dedx < 20)]
            
            if track_hits.empty:
                continue

            reco_p = calculate_hypfit_p(
                track_id=trk_id, 
                best_plane=2, 
                hit_df=df_hits, 
                target_pdg=211, 
                cleaning="all"
            )
            
            print(f"{label}, {reco_p}")

            # --- Optional Plotting Logic ---
            if do_plot:
                x, y = track_hits[rr_col].values, track_hits[dedx_col].values
                ax1.plot(x, y, label=f"{label.upper()} (p={reco_p:.2f})", color=color, 
                         linestyle=linestyle, marker='o', markersize=3, alpha=0.7)
                if label == 'cv':
                    cv_data = (x, y)
                    ax2.axhline(1, color='black', linewidth=1)
                elif cv_data is not None:
                    cv_interp = np.interp(x, cv_data[0], cv_data[1])
                    ax2.plot(x, y / cv_interp, color=color, linestyle=linestyle, marker='o', markersize=3)
        
        except KeyError:
            pass # Silent skip for batch processing

    if do_plot:
        ax1.set_title(f"Track Details: {trk_id[1]}")
        ax1.legend(fontsize=8)
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm.auto import tqdm # Progress bar

# 1. Initialize storage for momentum values
p_values = {
    'cv': [],
    'ccalp': [],
    'ccalm': []
}

unique_tracks = filtered_hit_dfs['cv'].index.droplevel(-1).unique()
rr_col = "rr"
dedx_col = "dedx"


#do_plot = (len(p_values['cv']) < 5) 
do_plot = True

# 2. Loop with progress bar (tqdm)
# Limit to 1000 tracks as requested
for trk_id in tqdm(unique_tracks[:500], desc="Processing Tracks"):
    
    cv_data = None
    # We only plot the first few to avoid memory/display issues

    if do_plot:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, 
                                       gridspec_kw={'height_ratios': [3, 1]})
        plt.subplots_adjust(hspace=0.05)

    for label, color, linestyle in zip(['cv', 'ccalp', 'ccalm'], 
                                       ['black', 'red', 'blue'],
                                       ['-', '--', '--']):
        df_hits = filtered_hit_dfs[label]
        try:
            track_hits = df_hits.loc[trk_id].sort_values(by=rr_col)
            track_hits = track_hits[(track_hits.dedx > 0.5) & (track_hits.dedx < 20)]
            
            if track_hits.empty:
                continue

            reco_p = calculate_hypfit_p(
                track_id=trk_id, 
                best_plane=2, 
                hit_df=df_hits, 
                target_pdg=211, 
                cleaning="all"
            )
            
            # Store value if within requested range [0.13, 1]
            if 0.13 <= reco_p <= 1.0:
                p_values[label].append(reco_p)

            # --- Optional Plotting Logic ---
            if do_plot:
                x, y = track_hits[rr_col].values, track_hits[dedx_col].values
                ax1.plot(x, y, label=f"{label.upper()} (p={reco_p:.2f})", color=color, 
                         linestyle=linestyle, marker='o', markersize=3, alpha=0.7)
                if label == 'cv':
                    cv_data = (x, y)
                    ax2.axhline(1, color='black', linewidth=1)
                elif cv_data is not None:
                    cv_interp = np.interp(x, cv_data[0], cv_data[1])
                    ax2.plot(x, y / cv_interp, color=color, linestyle=linestyle, marker='o', markersize=3)
        
        except KeyError:
            pass # Silent skip for batch processing

    if do_plot:
        ax1.set_title(f"Track Details: {trk_id[1]}")
        ax1.legend(fontsize=8)
        plt.show()

# 3. Plot the final 3 histograms
plt.figure(figsize=(10, 6))

for label, color in [('cv', 'black'), ('ccalp', 'red'), ('ccalm', 'blue')]:
    plt.hist(p_values[label], bins=40, range=(0.13, 1.0), histtype='step', 
             linewidth=2, color=color, label=label.upper())

plt.title("Reconstructed Momentum Distribution (0.13 - 1.0 GeV/c)", fontsize=14)
plt.xlabel("Reco Momentum [GeV/c]", fontsize=12)
plt.ylabel("Counts", fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()